# Day 18 — SQL Fundamentals Task

In this notebook, we execute **20 business queries** against the classic **Chinook Database** (SQLite format) using **DuckDB** to process analytical queries. This notebook forms the results deliverable alongside the raw `sql/queries.sql` file.

### Database Connection Setup
We connect to the SQLite database file (`chinook.db`) via DuckDB's native SQLite scanner extension. This allows us to query the SQLite tables directly using DuckDB's highly optimized vectorized engine.

In [3]:
import duckdb
import pandas as pd

# Connect to DuckDB
con = duckdb.connect()

# Install and load the sqlite extension to read chinook.db directly
con.execute("INSTALL sqlite; LOAD sqlite;")
con.execute("ATTACH 'chinook.db' AS chinook (TYPE SQLITE);")

# Set the default search schema to chinook to avoid prefixing tables
con.execute("SET schema = 'chinook';")

print("Connected successfully! Tables available in Chinook database:")
con.sql("SHOW TABLES;").show()

Connected successfully! Tables available in Chinook database:
┌───────────────┐
│     name      │
│    varchar    │
├───────────────┤
│ Album         │
│ Artist        │
│ Customer      │
│ Employee      │
│ Genre         │
│ Invoice       │
│ InvoiceLine   │
│ MediaType     │
│ Playlist      │
│ PlaylistTrack │
│ Track         │
└───────────────┘
     11 rows   



## 20 Business Queries

#### Query 1: Simple Filtering
Question: List all customers living in Canada.

In [2]:
q1 = """
SELECT CustomerId, FirstName, LastName, Country, Email
FROM Customer
WHERE Country = 'Canada';
"""
con.sql(q1).df()

,CustomerId,FirstName,LastName,Country,Email
0,3,François,Tremblay,Canada,ftremblay@gmail.com
1,14,Mark,Philips,Canada,mphilips12@shaw.ca
2,15,Jennifer,Peterson,Canada,jenniferp@rogers.ca
3,29,Robert,Brown,Canada,robbrown@shaw.ca
4,30,Edward,Francis,Canada,edfrancis@yachoo.ca
5,31,Martha,Silk,Canada,marthasilk@gmail.com
6,32,Aaron,Mitchell,Canada,aaronmitchell@yahoo.ca
7,33,Ellie,Sullivan,Canada,ellie.sullivan@shaw.ca


#### Query 2: Basic Aggregation
Question: Find the total revenue from all sales (invoice totals).

In [3]:
q2 = """
SELECT SUM(Total) as TotalRevenue
FROM Invoice;
"""
con.sql(q2).df()

,TotalRevenue
0,2328.6


#### Query 3: Group By & Having
Question: Find genres containing more than 100 tracks.

In [4]:
q3 = """
SELECT g.Name as GenreName, COUNT(t.TrackId) as TrackCount
FROM Genre g
JOIN Track t ON g.GenreId = t.GenreId
GROUP BY g.Name
HAVING COUNT(t.TrackId) > 100
ORDER BY TrackCount DESC;
"""
con.sql(q3).df()

,GenreName,TrackCount
0,Rock,1297
1,Latin,579
2,Metal,374
3,Alternative & Punk,332
4,Jazz,130


#### Query 4: Basic Join
Question: Retrieve the name of each artist along with their albums (first 20).

In [5]:
q4 = """
SELECT ar.Name as ArtistName, al.Title as AlbumTitle
FROM Artist ar
JOIN Album al ON ar.ArtistId = al.ArtistId
ORDER BY ArtistName, AlbumTitle
LIMIT 20;
"""
con.sql(q4).df()

,ArtistName,AlbumTitle
0,AC/DC,For Those About To Rock We Salute You
1,AC/DC,Let There Be Rock
2,Aaron Copland & London Symphony Orchestra,"A Copland Celebration, Vol. I"
3,Aaron Goldberg,Worlds
4,Academy of St. Martin in the Fields & Sir Nevi...,The World of Classical Favourites
5,Academy of St. Martin in the Fields Chamber En...,Sir Neville Marriner: A Celebration
6,"Academy of St. Martin in the Fields, John Birc...","Fauré: Requiem, Ravel: Pavane & Others"
7,"Academy of St. Martin in the Fields, Sir Nevil...",Bach: Orchestral Suites Nos. 1 - 4
8,Accept,Balls to the Wall
9,Accept,Restless and Wild


#### Query 5: Multiple Table Join
Question: Show invoice details including invoice ID, invoice date, customer full name, and billing country (first 20).

In [6]:
q5 = """
SELECT i.InvoiceId, i.InvoiceDate, c.FirstName || ' ' || c.LastName as CustomerName, i.BillingCountry
FROM Invoice i
JOIN Customer c ON i.CustomerId = c.CustomerId
ORDER BY i.InvoiceDate DESC
LIMIT 20;
"""
con.sql(q5).df()

,InvoiceId,InvoiceDate,CustomerName,BillingCountry
0,412,2025-12-22,Manoj Pareek,India
1,411,2025-12-14,Terhi Hämäläinen,Finland
2,410,2025-12-09,Madalena Sampaio,Portugal
3,409,2025-12-06,Robert Brown,Canada
4,408,2025-12-05,Victor Stevens,USA
5,407,2025-12-04,John Gordon,USA
6,406,2025-12-04,Kathy Chase,USA
7,405,2025-11-21,Dan Miller,USA
8,404,2025-11-13,Helena Holý,Czech Republic
9,403,2025-11-08,Diego Gutiérrez,Argentina


#### Query 6: Aggregated Join
Question: Calculate the total amount spent by each customer, sorted highest to lowest (top 10).

In [7]:
q6 = """
SELECT c.CustomerId, c.FirstName || ' ' || c.LastName as CustomerName, SUM(i.Total) as TotalSpent
FROM Customer c
JOIN Invoice i ON c.CustomerId = i.CustomerId
GROUP BY c.CustomerId, c.FirstName, c.LastName
ORDER BY TotalSpent DESC
LIMIT 10;
"""
con.sql(q6).df()

,CustomerId,CustomerName,TotalSpent
0,6,Helena Holý,49.62
1,26,Richard Cunningham,47.62
2,57,Luis Rojas,46.62
3,45,Ladislav Kovács,45.62
4,46,Hugh O'Reilly,45.62
5,28,Julia Barnett,43.62
6,24,Frank Ralston,43.62
7,37,Fynn Zimmermann,43.62
8,25,Victor Stevens,42.62
9,7,Astrid Gruber,42.62


#### Query 7: Left Join / Unmatched Rows
Question: Find all tracks that have never been purchased (first 20).

In [8]:
q7 = """
SELECT t.TrackId, t.Name as TrackName, t.Composer
FROM Track t
LEFT JOIN InvoiceLine il ON t.TrackId = il.TrackId
WHERE il.InvoiceLineId IS NULL
LIMIT 20;
"""
con.sql(q7).df()

,TrackId,TrackName,Composer
0,7,Let's Get It Up,"Angus Young, Malcolm Young, Brian Johnson"
1,11,C.O.D.,"Angus Young, Malcolm Young, Brian Johnson"
2,17,Let There Be Rock,AC/DC
3,18,Bad Boy Boogie,AC/DC
4,22,Whole Lotta Rosie,AC/DC
5,23,Walk On Water,"Steven Tyler, Joe Perry, Jack Blades, Tommy Shaw"
6,27,Dude (Looks Like A Lady),"Steven Tyler, Joe Perry, Desmond Child"
7,29,Cryin',"Steven Tyler, Joe Perry, Taylor Rhodes"
8,33,The Other Side,"Steven Tyler, Jim Vallance"
9,34,Crazy,"Steven Tyler, Joe Perry, Desmond Child"


#### Query 8: Ranked Join & Grouping
Question: Find the top 5 most popular artists by quantity of tracks sold.

In [9]:
q8 = """
SELECT ar.Name as ArtistName, SUM(il.Quantity) as TotalSold
FROM InvoiceLine il
JOIN Track t ON il.TrackId = t.TrackId
JOIN Album al ON t.AlbumId = al.AlbumId
JOIN Artist ar ON al.ArtistId = ar.ArtistId
GROUP BY ar.Name
ORDER BY TotalSold DESC
LIMIT 5;
"""
con.sql(q8).df()

,ArtistName,TotalSold
0,Iron Maiden,140.0
1,U2,107.0
2,Metallica,91.0
3,Led Zeppelin,87.0
4,Os Paralamas Do Sucesso,45.0


#### Query 9: Subquery (WHERE clause)
Question: Get the names of all employees who report directly to Andrew Adams.

In [10]:
q9 = """
SELECT EmployeeId, FirstName || ' ' || LastName as EmployeeName, Title
FROM Employee
WHERE ReportsTo = (SELECT EmployeeId FROM Employee WHERE FirstName = 'Andrew' AND LastName = 'Adams');
"""
con.sql(q9).df()

,EmployeeId,EmployeeName,Title
0,2,Nancy Edwards,Sales Manager
1,6,Michael Mitchell,IT Manager


#### Query 10: Correlated Subquery
Question: List each customer's invoice details for their maximum single-invoice purchase (first 20).

In [4]:
q10 = """
SELECT i1.CustomerId, i1.InvoiceId, i1.InvoiceDate, i1.Total
FROM Invoice i1
WHERE i1.Total = (
    SELECT MAX(i2.Total)
    FROM Invoice i2
    WHERE i2.CustomerId = i1.CustomerId
)
ORDER BY CustomerId
LIMIT 20;
"""
con.sql(q10).df()

,CustomerId,InvoiceId,InvoiceDate,Total
0,1,327,2024-12-07,13.86
1,2,12,2021-02-11,13.86
2,3,110,2022-04-21,13.86
3,4,208,2023-06-29,15.86
4,5,306,2024-09-05,16.86
5,6,404,2025-11-13,25.86
6,7,89,2022-01-18,18.86
7,8,187,2023-03-28,13.86
8,9,285,2024-06-04,13.86
9,10,383,2025-08-12,13.86


#### Query 11: Common Table Expression (CTE)
Question: Calculate the monthly sales revenue for the year 2011.

In [5]:
q11 = """
WITH Sales_2011 AS (
    SELECT InvoiceId, Total, strftime('%m', InvoiceDate) as MonthNum
    FROM Invoice
    WHERE strftime('%Y', InvoiceDate) = '2011'
)
SELECT MonthNum, SUM(Total) as MonthlyRevenue, COUNT(InvoiceId) as SalesCount
FROM Sales_2011
GROUP BY MonthNum
ORDER BY MonthNum;
"""
con.sql(q11).df()

,MonthNum,MonthlyRevenue,SalesCount


#### Query 12: Recursive CTE / Hierarchy
Question: Construct the organizational chart displaying employee names and their immediate supervisor's names.

In [13]:
q12 = """
WITH RECURSIVE OrgChart AS (
    SELECT EmployeeId, FirstName, LastName, ReportsTo, 1 as Level,
           FirstName || ' ' || LastName as Path
    FROM Employee
    WHERE ReportsTo IS NULL
    
    UNION ALL
    
    SELECT e.EmployeeId, e.FirstName, e.LastName, e.ReportsTo, oc.Level + 1,
           oc.Path || ' -> ' || e.FirstName || ' ' || e.LastName
    FROM Employee e
    JOIN OrgChart oc ON e.ReportsTo = oc.EmployeeId
)
SELECT Level, FirstName || ' ' || LastName as EmployeeName, Path
FROM OrgChart
ORDER BY Level, EmployeeId;
"""
con.sql(q12).df()

,Level,EmployeeName,Path
0,1,Andrew Adams,Andrew Adams
1,2,Nancy Edwards,Andrew Adams -> Nancy Edwards
2,2,Michael Mitchell,Andrew Adams -> Michael Mitchell
3,3,Jane Peacock,Andrew Adams -> Nancy Edwards -> Jane Peacock
4,3,Margaret Park,Andrew Adams -> Nancy Edwards -> Margaret Park
5,3,Steve Johnson,Andrew Adams -> Nancy Edwards -> Steve Johnson
6,3,Robert King,Andrew Adams -> Michael Mitchell -> Robert King
7,3,Laura Callahan,Andrew Adams -> Michael Mitchell -> Laura Call...


#### Query 13: Window Function (RANK)
Question: Rank customers within each country based on their total spend.

In [14]:
q13 = """
WITH CustomerSpent AS (
    SELECT c.CustomerId, c.FirstName || ' ' || c.LastName as CustomerName, c.Country, SUM(i.Total) as TotalSpent
    FROM Customer c
    JOIN Invoice i ON c.CustomerId = i.CustomerId
    GROUP BY c.CustomerId, c.FirstName, c.LastName, c.Country
)
SELECT Country, CustomerName, TotalSpent,
       RANK() OVER (PARTITION BY Country ORDER BY TotalSpent DESC) as SpendRank
FROM CustomerSpent
ORDER BY Country, SpendRank;
"""
con.sql(q13).df()

,Country,CustomerName,TotalSpent,SpendRank
0,Argentina,Diego Gutiérrez,37.62,1
1,Australia,Mark Taylor,37.62,1
2,Austria,Astrid Gruber,42.62,1
3,Belgium,Daan Peeters,37.62,1
4,Brazil,Luís Gonçalves,39.62,1
5,Brazil,Alexandre Rocha,37.62,2
6,Brazil,Fernanda Ramos,37.62,2
7,Brazil,Eduardo Martins,37.62,2
8,Brazil,Roberto Almeida,37.62,5
9,Canada,François Tremblay,39.62,1


#### Query 14: Window Function (LAG)
Question: Compare each invoice amount for Customer 1 with their previous invoice amount to show the difference.

In [15]:
q14 = """
SELECT CustomerId, InvoiceId, InvoiceDate, Total as CurrentInvoiceTotal,
       LAG(Total, 1) OVER (ORDER BY InvoiceDate) as PreviousInvoiceTotal,
       Total - LAG(Total, 1) OVER (ORDER BY InvoiceDate) as Difference
FROM Invoice
WHERE CustomerId = 1
ORDER BY InvoiceDate;
"""
con.sql(q14).df()

,CustomerId,InvoiceId,InvoiceDate,CurrentInvoiceTotal,PreviousInvoiceTotal,Difference
0,1,98,2022-03-11,3.98,NaN,NaN
1,1,121,2022-06-13,3.96,3.98,-0.02
2,1,143,2022-09-15,5.94,3.96,1.98
3,1,195,2023-05-06,0.99,5.94,-4.95
4,1,316,2024-10-27,1.98,0.99,0.99
5,1,327,2024-12-07,13.86,1.98,11.88
6,1,382,2025-08-07,8.91,13.86,-4.95


#### Query 15: Window Function (Running Total)
Question: Calculate the cumulative running total of sales over time by country.

In [16]:
q15 = """
SELECT BillingCountry, InvoiceDate, Total,
       SUM(Total) OVER (PARTITION BY BillingCountry ORDER BY InvoiceDate, InvoiceId) as CumulativeSales
FROM Invoice
ORDER BY BillingCountry, InvoiceDate;
"""
con.sql(q15).df()

,BillingCountry,InvoiceDate,Total,CumulativeSales
0,Argentina,2022-06-12,1.98,1.98
1,Argentina,2022-09-14,3.96,5.94
2,Argentina,2022-12-17,5.94,11.88
3,Argentina,2023-08-07,0.99,12.87
4,Argentina,2025-01-28,1.98,14.85
...,...,...,...,...
407,United Kingdom,2025-01-28,1.98,87.12
408,United Kingdom,2025-05-01,1.98,89.10
409,United Kingdom,2025-05-02,3.96,93.06
410,United Kingdom,2025-06-11,13.86,106.92


#### Query 16: Advanced Aggregation (GROUP BY CUBE equivalent)
Question: Count tracks by category combination of Genre and Media Type (first 25).

In [17]:
q16 = """
SELECT g.Name as Genre, m.Name as MediaType, COUNT(t.TrackId) as TrackCount
FROM Track t
JOIN Genre g ON t.GenreId = g.GenreId
JOIN MediaType m ON t.MediaTypeId = m.MediaTypeId
GROUP BY CUBE (g.Name, m.Name)
ORDER BY Genre NULLS LAST, MediaType NULLS LAST
LIMIT 25;
"""
con.sql(q16).df()

,Genre,MediaType,TrackCount
0,Alternative,Protected AAC audio file,38
1,Alternative,Protected MPEG-4 video file,1
2,Alternative,Purchased AAC audio file,1
3,Alternative,NaN,40
4,Alternative & Punk,MPEG audio file,332
5,Alternative & Punk,NaN,332
6,Blues,MPEG audio file,81
7,Blues,NaN,81
8,Bossa Nova,MPEG audio file,15
9,Bossa Nova,NaN,15


#### Query 17: String Manipulation
Question: List all track names in Playlist ID 1 as a single comma-separated string.

In [18]:
q17 = """
SELECT p.PlaylistId, p.Name as PlaylistName,
       string_agg(t.Name, ', ') as TrackList
FROM Playlist p
JOIN PlaylistTrack pt ON p.PlaylistId = pt.PlaylistId
JOIN Track t ON pt.TrackId = t.TrackId
WHERE p.PlaylistId = 1
GROUP BY p.PlaylistId, p.Name;
"""
con.sql(q17).df()

,PlaylistId,PlaylistName,TrackList
0,1,Music,"For Those About To Rock (We Salute You), Balls..."


#### Query 18: Date/Time Extraction
Question: Determine which day of the week generates the most invoice revenue.

In [19]:
q18 = """
SELECT dayname(InvoiceDate) as DayOfWeek,
       COUNT(InvoiceId) as SalesCount,
       SUM(Total) as TotalRevenue
FROM Invoice
GROUP BY DayOfWeek, dayofweek(InvoiceDate)
ORDER BY TotalRevenue DESC;
"""
con.sql(q18).df()

,DayOfWeek,SalesCount,TotalRevenue
0,Thursday,59,348.78
1,Monday,60,341.77
2,Friday,59,332.89
3,Sunday,58,329.76
4,Tuesday,59,326.82
5,Saturday,59,326.77
6,Wednesday,58,321.81


#### Query 19: Sales Rep Performance
Question: Compare total sales revenue managed by each Sales Support Agent (Employee).

In [20]:
q19 = """
SELECT e.EmployeeId, e.FirstName || ' ' || e.LastName as EmployeeName, e.Title,
       COALESCE(SUM(i.Total), 0) as TotalSalesManaged
FROM Employee e
LEFT JOIN Customer c ON e.EmployeeId = c.SupportRepId
LEFT JOIN Invoice i ON c.CustomerId = i.CustomerId
WHERE e.Title = 'Sales Support Agent'
GROUP BY e.EmployeeId, e.FirstName, e.LastName, e.Title
ORDER BY TotalSalesManaged DESC;
"""
con.sql(q19).df()

,EmployeeId,EmployeeName,Title,TotalSalesManaged
0,3,Jane Peacock,Sales Support Agent,833.04
1,4,Margaret Park,Sales Support Agent,775.40
2,5,Steve Johnson,Sales Support Agent,720.16


#### Query 20: Cohort Spending
Question: Group customers by the year of their first purchase to analyze their lifetime customer value.

In [21]:
q20 = """
WITH FirstPurchase AS (
    SELECT CustomerId, MIN(strftime('%Y', InvoiceDate)) as CohortYear
    FROM Invoice
    GROUP BY CustomerId
),
CustomerRevenue AS (
    SELECT CustomerId, SUM(Total) as TotalSpent
    FROM Invoice
    GROUP BY CustomerId
)
SELECT fp.CohortYear, COUNT(fp.CustomerId) as CohortSize, SUM(cr.TotalSpent) as CohortTotalSpend,
       AVG(cr.TotalSpent) as AverageSpentPerCustomer
FROM FirstPurchase fp
JOIN CustomerRevenue cr ON fp.CustomerId = cr.CustomerId
GROUP BY fp.CohortYear
ORDER BY CohortYear;
"""
con.sql(q20).df()

,CohortYear,CohortSize,CohortTotalSpend,AverageSpentPerCustomer
0,2021,46,1812.54,39.403043
1,2022,13,516.06,39.696923


## Validation

To satisfy the rubric, we perform validation checks to ensure no anomalous data exists (like NULL totals) and that our query environment correctly handles empty result sets.

#### Validation 1:
Verified no NULL invoice totals exist.

In [22]:
# Validation 1
# Expected: 0 rows
val1_query = """
SELECT *
FROM Invoice
WHERE Total IS NULL;
"""
con.sql(val1_query).df()

,InvoiceId,CustomerId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total


#### Validation 2:
Verified queries handle empty result sets correctly.

In [23]:
# Validation 2
# Expected: Empty result (0 rows)
val2_query = """
SELECT *
FROM Invoice
WHERE Total > 1000;
"""
con.sql(val2_query).df()

,InvoiceId,CustomerId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total


## Edge Cases

We implement queries handling two critical business database edge cases:

#### Edge Case 1: Null / Unassigned Foreign Keys (Graceful Defaults)
Query customers who don't have an assigned support representative. We use `COALESCE` to display a user-friendly default value when relations are missing.

In [24]:
# Edge Case 1: Unassigned Support Representatives
# Expected: 0 rows (all customers in Chinook have support reps, but query runs and handles nulls)
edge1_query = """
SELECT c.CustomerId, c.FirstName || ' ' || c.LastName as CustomerName,
       COALESCE(e.FirstName || ' ' || e.LastName, 'No Rep Assigned') as SupportRepName
FROM Customer c
LEFT JOIN Employee e ON c.SupportRepId = e.EmployeeId
WHERE c.SupportRepId IS NULL;
"""
con.sql(edge1_query).df()

,CustomerId,CustomerName,SupportRepName


#### Edge Case 2: Zero Sales / Unsold Entities (Missing Transactional Records)
Retrieve artists who have released albums but have recorded exactly zero tracks sold (unmatched in `InvoiceLine`).

In [25]:
# Edge Case 2: Artists with Albums but Zero Sales
# Expected: List of artists with 0 total tracks sold
edge2_query = """
SELECT ar.ArtistId, ar.Name as ArtistName, COUNT(il.InvoiceLineId) as TracksSold
FROM Artist ar
LEFT JOIN Album al ON ar.ArtistId = al.ArtistId
LEFT JOIN Track t ON al.AlbumId = t.AlbumId
LEFT JOIN InvoiceLine il ON t.TrackId = il.TrackId
GROUP BY ar.ArtistId, ar.Name
HAVING COUNT(il.InvoiceLineId) = 0
ORDER BY ArtistName
LIMIT 10;
"""
con.sql(edge2_query).df()

,ArtistId,ArtistName,TracksSold
0,43,A Cor Do Som,0
1,230,Aaron Copland & London Symphony Orchestra,0
2,202,Aaron Goldberg,0
3,215,Academy of St. Martin in the Fields Chamber En...,0
4,239,"Academy of St. Martin in the Fields, Sir Nevil...",0
5,161,Aerosmith & Sierra Leone's Refugee Allstars,0
6,197,Aisha Duo,0
7,206,Alberto Turco & Nova Schola Gregoriana,0
8,209,"Anne-Sophie Mutter, Herbert Von Karajan & Wien...",0
9,166,Avril Lavigne,0


## Self Reflection

### What went well:
- Successfully connected a SQLite database to DuckDB using ATTACH.
- Applied a wide range of SQL techniques including joins, CTEs, recursive CTEs, and window functions.
- Structured queries around business questions rather than isolated SQL syntax.

### Challenges:
- Understanding date functions across DuckDB and SQLite.
- Designing meaningful business questions for more advanced SQL features.

### Key Takeaways:
- SQL is most powerful when used to answer business questions, not just retrieve data.
- Window functions provide analytical capabilities that are difficult to reproduce with simple GROUP BY statements.
- DuckDB makes local analytical workflows extremely fast and convenient.